In [ ]:
!pip install groq

In [ ]:
import json
import random
from datetime import datetime

def filter_and_sample_results(input_file, output_file, sample_size=50):
    """
    Filter out unscored results, shuffle, and sample N entries
    """

    # Load the data
    with open(input_file, 'r') as f:
        data = json.load(f)

    # Get all results
    all_results = data.get("results", [])

    # Filter criteria: Keep results that have at least one gold score
    # Check for any field ending with '_gold'
    filtered_results = []
    for result in all_results:
        has_gold_score = False
        for key in result.keys():
            if key.endswith('_gold'):
                # Check if it's a numeric value
                value = result[key]
                if isinstance(value, (int, float)):
                    has_gold_score = True
                    break

        if has_gold_score:
            filtered_results.append(result)

    print(f"Original results: {len(all_results)}")
    print(f"Results with gold scores: {len(filtered_results)}")
    print(f"Results removed: {len(all_results) - len(filtered_results)}")

    # Random shuffle
    random.seed(42)  # For reproducibility
    random.shuffle(filtered_results)

    # Sample N entries (or all if less than N)
    if len(filtered_results) <= sample_size:
        sampled_results = filtered_results
        print(f"Sampling all {len(filtered_results)} available results")
    else:
        sampled_results = filtered_results[:sample_size]
        print(f"Sampled {sample_size} out of {len(filtered_results)} available results")

    # Update metadata
    metadata = data.get("metadata", {})
    metadata.update({
        "total_queries": len(sampled_results),
        "timestamp": int(datetime.now().timestamp()),
        "filtered": True,
        "original_total": len(all_results),
        "filtered_total": len(filtered_results),
        "sampled_total": len(sampled_results),
        "sample_size": sample_size,
        "shuffled": True,
        "random_seed": 42
    })

    # Remove success_rate if it's no longer accurate
    if "success_rate" in metadata:
        del metadata["success_rate"]

    # Create new data structure
    new_data = {
        "metadata": metadata,
        "results": sampled_results
    }

    # Save to output file
    with open(output_file, 'w') as f:
        json.dump(new_data, f, indent=2)

    print(f"\nFiltered and sampled data saved to: {output_file}")

    # Show some statistics about the sampled data
    print("\nSample Statistics:")
    print("-" * 40)

    # Count by domain
    domain_counts = {}
    for result in sampled_results:
        domain = result.get("domain", "Unknown")
        domain_counts[domain] = domain_counts.get(domain, 0) + 1

    print("Domain distribution:")
    for domain, count in domain_counts.items():
        print(f"  {domain}: {count}")

    # Count by difficulty
    difficulty_counts = {}
    for result in sampled_results:
        difficulty = result.get("difficulty", "Unknown")
        difficulty_counts[difficulty] = difficulty_counts.get(difficulty, 0) + 1

    print("\nDifficulty distribution:")
    for difficulty, count in difficulty_counts.items():
        print(f"  {difficulty}: {count}")

    # Show first few entries as preview
    print(f"\nFirst 3 entries in sample:")
    print("-" * 40)
    for i, result in enumerate(sampled_results[:3]):
        print(f"\nEntry {i+1}:")
        print(f"  Domain: {result.get('domain')}")
        print(f"  Business Event: {result.get('business_event')}")
        print(f"  Difficulty: {result.get('difficulty')}")
        print(f"  Follow-up Type: {result.get('follow_up_type')}")

    return new_data

# Usage example
if __name__ == "__main__":
    input_file = "Gold_Task2.json"  # Change this to your actual file name
    output_file = "gold.json"

    # Process the data
    processed_data = filter_and_sample_results(input_file, output_file, sample_size=50)

In [ ]:
INTENT_COMPLETENESS_PROMPT = """
You are an expert evaluator for contextual follow-up queries in conversational analytics.

Your goal is to evaluate how completely and clearly a follow-up query expresses its analytical intent, scope, and contextual dependency on Task 1.

---

SCORING RUBRIC (0 to 10 points):
10 - Perfect Intent Expression: Clear analytic verb, specific goal, explicit Task 1 dependency, and complete context
9-8 - Excellent: Clear intent with good Task 1 linkage and specific analytical goal
7-6 - Good: Clear analytic intent but missing some contextual references to Task 1
5-4 - Fair: Basic analytic intent but vague or weak Task 1 dependency
3-1 - Poor: Unclear, fragmented, or lacks connection to Task 1
0 - No actionable intent or completely independent of Task 1

KEY CRITERIA FOR FOLLOW-UP QUERIES:
- Analytic Verb: explain, identify, compare, analyze, how, what, why, etc.
- Clear Goal: What is being analyzed building on Task 1 findings
- Task 1 Dependency: Explicit or clear reference to Task 1 results/insights
- Sufficient Context: Domain, segments, parameters, or analytical dimensions
- Actionable: Can be executed as a logical follow-up to Task 1
- Complete: Forms a self-contained analytical progression

---

EXAMPLES WITH SCORES:

10/10: "After identifying the most frequent agent action in Task 1, what is the average time interval between that action and cancellation events across different flight routes?"
9/10: "How does the frequency of denials attributed to the most common policy clause differ across ticket classes?"
7/10: "What is the distribution of outcomes for the top agent action identified earlier?"
5/10: "Analyze cancellation patterns by flight route"
3/10: "Time intervals for agent actions"
0/10: "cancellation data"

---

OUTPUT FORMAT:
Return your evaluation strictly as JSON format:

{
  "follow_up_query": "<the follow-up query>",
  "task1_query": "<the original task1 query>",
  "intent_score": <0 to 10>,
  "contextual_clarity": <0 to 10>,
  "reasoning": "<1-2 sentences explaining the score>",
  "issues_detected": ["missing_task1_reference", "vague_goal", "weak_context", "no_analytic_verb", "none"]
}

---

Now evaluate this query:
Follow-up Query: "{follow_up_query}"
Task1 Query: "{task1_query}"

Return JSON only.
"""

In [ ]:
REALISM_FEASIBILITY_PROMPT = """
You are an evaluator for enterprise conversational analytics systems, specifically for contextual follow-up queries.

Your task is to assess each follow-up query for three dimensions:
1. Domain Realism (10 points)
2. Scenario Feasibility (10 points)
3. Contextual Consistency (10 points)

---

DOMAIN REALISM (10 POINTS)
Evaluate whether the query naturally belongs to the call-center analytics domain and maintains domain consistency with Task 1.

Scoring Guide:
10 - Perfect Domain Realism: Clearly continues Task 1 domain context with authentic terminology
7 - Reasonable: Maintains domain context but uses somewhat generic language
4 - Weak: Loose domain connection or inconsistent with Task 1 domain
0 - Domain Mismatch: Completely different domain from Task 1

---

SCENARIO FEASIBILITY (10 POINTS)
Evaluate whether the follow-up scenario could realistically build on Task 1 findings and be analyzed using available data.

Scoring Guide:
10 - Fully Feasible: Logical progression from Task 1, analyzable with call data
7 - Mostly Feasible: Plausible progression but relies on abstract variables
4 - Borderline: Conceptually related but includes non-measurable elements
0 - Not Feasible: Impossible progression or fictional scenario

---

CONTEXTUAL CONSISTENCY (10 POINTS)
Evaluate how well the follow-up query logically builds upon and extends the Task 1 analysis.

Scoring Guide:
10 - Perfect Consistency: Direct logical progression building on Task 1 findings
7 - Good Consistency: Reasonable extension of Task 1 themes
4 - Weak Consistency: Loosely related to Task 1 but not a direct progression
0 - No Consistency: Unrelated or contradictory to Task 1 analysis

---

OUTPUT FORMAT:
Return your evaluation strictly as JSON format:

{
  "follow_up_query": "<the follow-up query>",
  "task1_query": "<the original task1 query>",
  "domain_realism_score": <0 to 10>,
  "scenario_feasibility_score": <0 to 10>,
  "contextual_consistency_score": <0 to 10>,
  "reasoning": {
    "domain_realism": "<1-2 sentences>",
    "feasibility": "<1-2 sentences>",
    "contextual_consistency": "<1-2 sentences>"
  },
  "issues_detected": ["domain_mismatch", "unrealistic_progression", "weak_context_link", "none"]
}

---

Example Input:
Follow-up Query: "What is the average time lag between the most frequent agent action identified in Task 1 and actual flight service cancellation?"
Task1 Query: "Which agent action most frequently precedes a flight service cancellation?"

Example JSON Output:
{
  "follow_up_query": "What is the average time lag between the most frequent agent action identified in Task 1 and actual flight service cancellation?",
  "task1_query": "Which agent action most frequently precedes a flight service cancellation?",
  "domain_realism_score": 10,
  "scenario_feasibility_score": 10,
  "contextual_consistency_score": 10,
  "reasoning": {
    "domain_realism": "Maintains perfect flight service domain consistency with authentic operational metrics.",
    "feasibility": "Time interval analysis is fully feasible with call timestamp data.",
    "contextual_consistency": "Direct logical progression from identifying frequent actions to analyzing their temporal characteristics."
  },
  "issues_detected": ["none"]
}

Now evaluate this query:
Follow-up Query: "{follow_up_query}"
Task1 Query: "{task1_query}"
"""

In [ ]:
DIFFICULTY_PROMPT = """
You are an expert evaluator of follow-up query difficulty in call-center analytics.

Evaluate if the assigned difficulty matches the query's complexity level.

DIFFICULTY LEVELS:
- Easy: Direct extension of Task 1, basic retrieval/segmentation
- Medium: Moderate reasoning building on Task 1, single-step analysis
- Hard: Complex multi-dimensional reasoning, counterfactual/predictive analysis

SCORING (0-10):
10 - Perfect match
7-9 - Strong match with minor issues
4-6 - Moderate mismatch
0-3 - Wrong difficulty

OUTPUT JSON:
{
  "follow_up_query": "<query>",
  "task1_query": "<task1>",
  "assigned_difficulty": "<easy|medium|hard>",
  "difficulty_score": 0-10,
  "evaluation_reason": "<brief explanation>"
}

FOLLOW-UP QUERY: {follow_up_query}
TASK1 QUERY: {task1_query}
ASSIGNED DIFFICULTY: {assigned_difficulty}

Return JSON only:
"""

In [ ]:
FOLLOW_UP_TYPE_PROMPT = """
You are an expert evaluator of follow-up query categorization in analytical systems.

Evaluate if the assigned follow-up type correctly describes the query's analytical approach.

FOLLOW-UP TYPES:
- Evidence Retrieval: Requests specific data samples, cases, or evidence
- Clarification / Drill-Down: Seeks deeper details or clarification on Task 1 findings
- Contextual Follow-Up: Adds contextual dimensions (segments, timeframes, comparisons)
- Refinement Queries: Improves or refines Task 1 analysis with additional parameters
- Counterfactual / Exception Follow-Up: Explores what-if scenarios or exception cases

SCORING (0-10):
10 - Perfect categorization
7-9 - Good match with minor issues
4-6 - Partial match but could be better categorized
0-3 - Wrong categorization

OUTPUT JSON:
{
  "follow_up_query": "<query>",
  "task1_query": "<task1>",
  "assigned_follow_up_type": "<assigned type>",
  "correct_follow_up_type": "<detected correct type>",
  "type_match_score": 0-10,
  "evaluation_reason": "<brief explanation>"
}

FOLLOW-UP QUERY: {follow_up_query}
TASK1 QUERY: {task1_query}
ASSIGNED FOLLOW-UP TYPE: {follow_up_type}

Return JSON only:
"""

In [ ]:
CLARITY_SPECIFICITY_PROMPT = """
You are an expert evaluator of follow-up query quality in call-center analytics.

Evaluate each follow-up query on:
1. Clarity (10 points): Grammatical correctness, professional phrasing, clear structure
2. Specificity (10 points): Clear analytical scope, context parameters, measurable focus

SCORING (0-10 per dimension):
10 - Excellent, 7-9 - Good, 4-6 - Fair, 0-3 - Poor

OUTPUT JSON:
{
  "follow_up_query": "<query>",
  "task1_query": "<task1>",
  "clarity_score": 0-10,
  "specificity_score": 0-10,
  "reasoning": {
    "clarity": "<brief clarity assessment>",
    "specificity": "<brief specificity assessment>"
  }
}

FOLLOW-UP QUERY: {follow_up_query}
TASK1 QUERY: {task1_query}

Return JSON only:
"""

In [ ]:
SAFETY_BIAS_PROMPT = """
You are an enterprise ethics auditor evaluating follow-up analytical queries.

Evaluate on two dimensions:
SAFETY (0-5): No PII requests, compliance with regulations, no harmful language
FAIRNESS (0-5): Neutral phrasing, no bias toward groups, objective treatment

FINAL SCORE: Safety + Fairness (0-10)

OUTPUT JSON:
{
  "follow_up_query": "<query>",
  "task1_query": "<task1>",
  "safety_score": 0-5,
  "fairness_score": 0-5,
  "total_score": 0-10,
  "explanation": "<combined rationale>"
}

FOLLOW-UP QUERY: {follow_up_query}
TASK1 QUERY: {task1_query}

Return JSON only:
"""

In [ ]:
import json, time, random
import numpy as np
from typing import List, Dict, Any
from tqdm import tqdm
from groq import Groq
from sklearn.linear_model import LinearRegression

# ========= CONFIGURATION =========
MODEL_NAME = "moonshotai/kimi-k2-instruct-0905"
QUERY_LIMIT = 100
ENABLE_CALIBRATION = True  # Disable if no gold data, enable if have gold data

# Add more keys for fallback
API_KEYS = [
    "gsk_XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX",
]

# ========= JUDGES CONFIGURATION =========
JUDGES_CONFIG = {
    "difficulty": DIFFICULTY_PROMPT,
    "follow_up_type": FOLLOW_UP_TYPE_PROMPT,
    "intent_completeness": INTENT_COMPLETENESS_PROMPT,
    "realism_feasibility": REALISM_FEASIBILITY_PROMPT,
    "clarity_specificity": CLARITY_SPECIFICITY_PROMPT,
    "safety_bias": SAFETY_BIAS_PROMPT
}

# ========= TEST API KEYS =========
def test_api_keys():
    """Test if API keys are working with the new model"""
    print(" TESTING API KEYS...")

    working_keys = []

    for i, key in enumerate(API_KEYS):
        try:
            client = Groq(api_key=key)
            response = client.chat.completions.create(
                model=MODEL_NAME,
                messages=[{"role": "user", "content": "Say 'OK' in JSON: {\"status\": \"ok\"}"}],
                temperature=0.0,
                max_tokens=50,
                response_format={"type": "json_object"}
            )
            result = response.choices[0].message.content
            if "ok" in result.lower():
                working_keys.append(key)
                print(f" Key {i}: WORKING")
            else:
                print(f" Key {i}: RESPONSE ISSUE")
        except Exception as e:
            print(f" Key {i}: FAILED - {str(e)}")

    print(f" Result: {len(working_keys)}/{len(API_KEYS)} keys available")
    return working_keys

# ========= OPTIMIZED GROQ CLIENT =========
class OptimizedGroqClient:
    def __init__(self, api_keys: List[str], judges: List[str]):
        self.api_keys = api_keys
        self.judges = judges
        self.current_key_index = 0

        # Initialize clients with rate limiting
        self.clients = {}
        for i, key in enumerate(api_keys):
            self.clients[f"key_{i}"] = {
                'client': Groq(api_key=key),
                'last_call_time': 0,
                'consecutive_failures': 0
            }

        # Rate limiting based on 30 requests/minute
        self.min_call_interval = 2.1  # 2.1 seconds between calls (28/min for safety)
        print(f" OPTIMIZED GROQ SYSTEM: {len(api_keys)} API keys for {len(judges)} judges")
        print(f" Using model: {MODEL_NAME}")

    def _get_next_key(self):
        """Smart key rotation with failure tracking"""
        original_index = self.current_key_index

        for _ in range(len(self.api_keys)):
            key_id = f"key_{self.current_key_index}"
            self.current_key_index = (self.current_key_index + 1) % len(self.api_keys)

            # Skip keys with too many consecutive failures
            client_data = self.clients[key_id]
            if client_data.get('consecutive_failures', 0) > 3:
                continue

            return key_id, client_data

        # If all keys have issues, reset and use any
        self.current_key_index = (original_index + 1) % len(self.api_keys)
        key_id = f"key_{self.current_key_index}"
        return key_id, self.clients[key_id]

    def extract_query_data(self, query_json: Dict[str, Any]) -> Dict[str, Any]:
        """Extract query data from Task 2 schema"""
        return {
            "follow_up_query": query_json.get("follow_up_query", ""),
            "task1_query": query_json.get("task1_query", ""),
            "difficulty": query_json.get("difficulty", "UNKNOWN"),
            "follow_up_type": query_json.get("follow_up_type", ""),
            "domain": query_json.get("domain", ""),
            "business_event": query_json.get("business_event", ""),
            "analytical_progression": query_json.get("analytical_progression", ""),
            "context_dependency": query_json.get("context_dependency", "")
        }

    def call_completion(self, judge_name: str, query_data: Dict[str, Any]) -> Dict[str, Any]:
        """Make API call with Task 2 query data"""
        key_id, client_data = self._get_next_key()
        client = client_data['client']

        for attempt in range(3):  # 3 retries
            try:
                # Rate limiting
                current_time = time.time()
                time_since_last_call = current_time - client_data['last_call_time']
                delay = max(0, self.min_call_interval - time_since_last_call)
                if delay > 0:
                    time.sleep(delay)

                # Extract Task 2 data
                follow_up_query = query_data.get("follow_up_query", "")
                task1_query = query_data.get("task1_query", "")
                assigned_difficulty = query_data.get("difficulty", "UNKNOWN")
                follow_up_type = query_data.get("follow_up_type", "")

                # Create prompt based on judge
                prompt_template = JUDGES_CONFIG[judge_name]

                if judge_name == "difficulty":
                    prompt = prompt_template.replace("{follow_up_query}", follow_up_query)\
                                           .replace("{task1_query}", task1_query)\
                                           .replace("{assigned_difficulty}", assigned_difficulty)
                elif judge_name == "follow_up_type":
                    prompt = prompt_template.replace("{follow_up_query}", follow_up_query)\
                                           .replace("{task1_query}", task1_query)\
                                           .replace("{follow_up_type}", follow_up_type)
                elif judge_name in ["intent_completeness", "realism_feasibility",
                                  "clarity_specificity", "safety_bias"]:
                    # All other prompts need both queries
                    prompt = prompt_template.replace("{follow_up_query}", follow_up_query)\
                                           .replace("{task1_query}", task1_query)
                else:
                    prompt = prompt_template.replace("{query}", follow_up_query)

                print(f"   {judge_name} | Key: {key_id} | Attempt: {attempt + 1}")

                # API call
                response = client.chat.completions.create(
                    model=MODEL_NAME,
                    messages=[{"role": "user", "content": prompt}],
                    temperature=0.0,
                    max_tokens=300,
                    response_format={"type": "json_object"}
                )

                client_data['last_call_time'] = time.time()
                client_data['consecutive_failures'] = 0  # Reset failure count

                text = response.choices[0].message.content

                # Parse response
                parsed = json.loads(text)
                return parsed

            except Exception as e:
                error_msg = str(e)
                print(f" {judge_name} attempt {attempt + 1} failed: {error_msg}")

                # Track failures
                client_data['consecutive_failures'] = client_data.get('consecutive_failures', 0) + 1

                if attempt < 2:  # Not the last attempt
                    wait_time = (2 ** attempt) + random.uniform(1, 3)
                    print(f" Waiting {wait_time:.1f}s before retry...")
                    time.sleep(wait_time)
                else:
                    print(f" All retries failed for {judge_name}")
                    return {"error": f"API call failed: {error_msg}", "score": 0.0}

        return {"error": "Max retries exceeded", "score": 0.0}

# ========= CALIBRATION SYSTEM =========
class CalibrationSystem:
    def __init__(self):
        self.calibration_params = {}
        self.calibration_metrics = {}
        self.is_calibrated = False
        self.calibration_comparison_data = []

    def calibrate_with_gold_data(self, gold_queries: List[Dict], evaluator) -> Dict:
        """Calibrate using gold dataset"""
        print(" CALIBRATION PHASE: LLM judging gold queries...")

        # Step 1: Have LLM judge the gold queries
        llm_scores_on_gold = []
        for i, query in enumerate(gold_queries):
            print(f" Calibrating with gold query {i+1}/{len(gold_queries)}: {query.get('follow_up_query', '')[:50]}...")
            llm_score = evaluator.evaluate_single_query(query)
            llm_scores_on_gold.append(llm_score)

        # Step 2: Extract human gold scores from the same queries
        human_scores = []
        for query in gold_queries:
            human_score = {
                "difficulty_gold": query.get("difficulty_gold", 0),
                "intent_completeness_gold": query.get("intent_completeness_gold", 0),
                "realism_feasibility_realism_gold": query.get("realism_feasibility_realism_gold", 0),
                "realism_feasibility_feasibility_gold": query.get("realism_feasibility_feasibility_gold", 0),
                "clarity_specificity_clarity_gold": query.get("clarity_specificity_clarity_gold", 0),
                "clarity_specificity_specificity_gold": query.get("clarity_specificity_specificity_gold", 0),
                "follow_up_type_match_gold": query.get("follow_up_type_match_gold", 0),
                "safety_bias_safety_gold": query.get("safety_bias_safety_gold", 0),
                "safety_bias_bias_gold": query.get("safety_bias_bias_gold", 0)
            }
            human_scores.append(human_score)

        # Step 3: Store comparison data for verification
        self._store_calibration_comparison(gold_queries, llm_scores_on_gold, human_scores)

        # Step 4: Calculate calibration metrics
        calibration_metrics = self.calculate_calibration_metrics(llm_scores_on_gold, human_scores)

        # Step 5: Train calibration models
        self.train_calibration_models(llm_scores_on_gold, human_scores)

        self.is_calibrated = True
        return calibration_metrics

    def calculate_calibration_metrics(self, llm_scores: List[Dict], human_scores: List[Dict]) -> Dict:
        """Calculate how well LLM scores match human scores"""
        metrics_to_calibrate = [
            'difficulty', 'intent_completeness',
            'realism_feasibility_realism', 'realism_feasibility_feasibility',
            'clarity_specificity_clarity', 'clarity_specificity_specificity',
            'follow_up_type_match', 'safety_bias_safety', 'safety_bias_bias'
        ]

        calibration_results = {}

        for metric in metrics_to_calibrate:
            llm_values = []
            human_values = []

            for llm_score, human_score in zip(llm_scores, human_scores):
                # Get LLM raw score
                llm_key = f"{metric}_raw"
                llm_val = llm_score.get(llm_key)

                # Get human gold score
                human_key = f"{metric}_gold"
                human_val = human_score.get(human_key)

                if llm_val is not None and human_val is not None:
                    llm_values.append(float(llm_val))
                    human_values.append(float(human_val))

            if len(llm_values) >= 3:
                llm_arr = np.array(llm_values)
                human_arr = np.array(human_values)

                # Calculate metrics
                correlation = np.corrcoef(llm_arr, human_arr)[0, 1] if np.std(llm_arr) > 0 and np.std(human_arr) > 0 else 0
                bias = np.mean(llm_arr) - np.mean(human_arr)
                mae = np.mean(np.abs(llm_arr - human_arr))
                rmse = np.sqrt(np.mean((llm_arr - human_arr) ** 2))

                calibration_results[metric] = {
                    'correlation': float(correlation),
                    'bias': float(bias),
                    'mae': float(mae),
                    'rmse': float(rmse),
                    'llm_mean': float(np.mean(llm_arr)),
                    'human_mean': float(np.mean(human_arr)),
                    'sample_size': len(llm_values)
                }

        return calibration_results

    def train_calibration_models(self, llm_scores: List[Dict], human_scores: List[Dict]):
        """Train calibration models using linear regression"""
        metrics_to_calibrate = [
            'difficulty', 'intent_completeness',
            'realism_feasibility_realism', 'realism_feasibility_feasibility',
            'clarity_specificity_clarity', 'clarity_specificity_specificity',
            'follow_up_type_match', 'safety_bias_safety', 'safety_bias_bias'
        ]

        for metric in metrics_to_calibrate:
            llm_values = []
            human_values = []

            for llm_score, human_score in zip(llm_scores, human_scores):
                llm_key = f"{metric}_raw"
                llm_val = llm_score.get(llm_key)

                human_key = f"{metric}_gold"
                human_val = human_score.get(human_key)

                if llm_val is not None and human_val is not None:
                    llm_values.append(float(llm_val))
                    human_values.append(float(human_val))

            if len(llm_values) >= 5:
                X = np.array(llm_values).reshape(-1, 1)
                y = np.array(human_values)

                model = LinearRegression()
                model.fit(X, y)

                self.calibration_params[metric] = {
                    'slope': float(model.coef_[0]),
                    'intercept': float(model.intercept_),
                    'r_squared': float(model.score(X, y))
                }

    def _store_calibration_comparison(self, gold_queries: List[Dict], llm_scores: List[Dict], human_scores: List[Dict]):
        """Store LLM vs Human scores for verification"""
        self.calibration_comparison_data = []

        for i, (query, llm_score, human_score) in enumerate(zip(gold_queries, llm_scores, human_scores)):
            comparison_entry = {
                "query_id": i,
                "follow_up_query": query.get("follow_up_query", ""),
                "task1_query": query.get("task1_query", ""),
                "domain": query.get("domain", ""),
                "difficulty": {
                    "llm_raw": llm_score.get("difficulty_raw"),
                    "human_gold": human_score.get("difficulty_gold"),
                    "difference": llm_score.get("difficulty_raw", 0) - human_score.get("difficulty_gold", 0)
                },
                "intent_completeness": {
                    "llm_raw": llm_score.get("intent_completeness_raw"),
                    "human_gold": human_score.get("intent_completeness_gold"),
                    "difference": llm_score.get("intent_completeness_raw", 0) - human_score.get("intent_completeness_gold", 0)
                },
                "realism_feasibility_realism": {
                    "llm_raw": llm_score.get("realism_feasibility_realism_raw"),
                    "human_gold": human_score.get("realism_feasibility_realism_gold"),
                    "difference": llm_score.get("realism_feasibility_realism_raw", 0) - human_score.get("realism_feasibility_realism_gold", 0)
                },
                "realism_feasibility_feasibility": {
                    "llm_raw": llm_score.get("realism_feasibility_feasibility_raw"),
                    "human_gold": human_score.get("realism_feasibility_feasibility_gold"),
                    "difference": llm_score.get("realism_feasibility_feasibility_raw", 0) - human_score.get("realism_feasibility_feasibility_gold", 0)
                },
                "clarity_specificity_clarity": {
                    "llm_raw": llm_score.get("clarity_specificity_clarity_raw"),
                    "human_gold": human_score.get("clarity_specificity_clarity_gold"),
                    "difference": llm_score.get("clarity_specificity_clarity_raw", 0) - human_score.get("clarity_specificity_clarity_gold", 0)
                },
                "clarity_specificity_specificity": {
                    "llm_raw": llm_score.get("clarity_specificity_specificity_raw"),
                    "human_gold": human_score.get("clarity_specificity_specificity_gold"),
                    "difference": llm_score.get("clarity_specificity_specificity_raw", 0) - human_score.get("clarity_specificity_specificity_gold", 0)
                },
                "follow_up_type_match": {
                    "llm_raw": llm_score.get("follow_up_type_match_raw"),
                    "human_gold": human_score.get("follow_up_type_match_gold"),
                    "difference": llm_score.get("follow_up_type_match_raw", 0) - human_score.get("follow_up_type_match_gold", 0)
                },
                "safety_bias_safety": {
                    "llm_raw": llm_score.get("safety_bias_safety_raw"),
                    "human_gold": human_score.get("safety_bias_safety_gold"),
                    "difference": llm_score.get("safety_bias_safety_raw", 0) - human_score.get("safety_bias_safety_gold", 0)
                },
                "safety_bias_bias": {
                    "llm_raw": llm_score.get("safety_bias_bias_raw"),
                    "human_gold": human_score.get("safety_bias_bias_gold"),
                    "difference": llm_score.get("safety_bias_bias_raw", 0) - human_score.get("safety_bias_bias_gold", 0)
                }
            }
            self.calibration_comparison_data.append(comparison_entry)

    def save_calibration_verification(self, filename: str = "calibration_verification.json"):
        """Save the LLM vs Human comparison for verification"""
        if not self.calibration_comparison_data:
            print(" No calibration data to save")
            return

        verification_data = {
            "metadata": {
                "timestamp": int(time.time()),
                "total_queries": len(self.calibration_comparison_data),
                "calibration_applied": self.is_calibrated,
                "calibration_params": self.calibration_params
            },
            "calibration_metrics": self.calibration_metrics,
            "llm_vs_human_comparison": self.calibration_comparison_data
        }

        with open(filename, 'w') as f:
            json.dump(verification_data, f, indent=2)

        print(f" Calibration verification saved to: {filename}")

    def apply_calibration(self, llm_score: Dict) -> Dict:
        """Apply calibration to a single LLM score"""
        if not self.is_calibrated:
            return llm_score

        calibrated_score = llm_score.copy()

        for metric, params in self.calibration_params.items():
            llm_key = f"{metric}_raw"
            raw_value = llm_score.get(llm_key)

            if raw_value is not None:
                # Apply calibration: calibrated = slope * raw + intercept
                calibrated_value = params['slope'] * raw_value + params['intercept']
                calibrated_value = max(0, min(10, calibrated_value))  # Clip to 0-10

                calibrated_score[f"{metric}_calibrated"] = calibrated_value
                calibrated_score[f"{metric}_raw_uncalibrated"] = raw_value

        return calibrated_score

# ========= OPTIMIZED EVALUATION SYSTEM =========
class OptimizedEvaluationSystem:
    def __init__(self, api_keys: List[str], judges_config: Dict[str, str]):
        self.judges_config = judges_config
        self.judge_names = list(judges_config.keys())
        self.client = OptimizedGroqClient(api_keys, self.judge_names)

        self.stats = {
            'total_queries': 0,
            'successful_judgments': 0,
            'failed_judgments': 0,
            'total_api_calls': 0,
            'start_time': time.time()
        }

        print(f" OPTIMIZED EVALUATION SYSTEM")
        print(f"   Judges: {len(self.judge_names)}")
        print(f"   Target: {QUERY_LIMIT} queries only")
        print(f"   Model: {MODEL_NAME}")

    def safe_float(self, x, default=0.0):
        """Safe float conversion"""
        try:
            return float(x)
        except:
            return default

    def safe_str(self, x, default=""):
        """Safe string conversion"""
        if isinstance(x, dict):
            return json.dumps(x)
        return str(x) if x else default

    def evaluate_single_judge(self, judge_name: str, query_data: Dict[str, Any]) -> Dict[str, Any]:
        """Evaluate one query with one judge - UPDATED for Task 2"""
        self.stats['total_api_calls'] += 1

        try:
            result = self.client.call_completion(judge_name, query_data)

            if "error" in result:
                self.stats['failed_judgments'] += 1
                return {
                    f"{judge_name}_raw": 0.0,
                    f"{judge_name}_expl": f"Error: {result['error']}"
                }
            else:
                self.stats['successful_judgments'] += 1

                output = {}
                if judge_name == "realism_feasibility":
                    output.update({
                        "realism_feasibility_realism_raw": self.safe_float(result.get("domain_realism_score")),
                        "realism_feasibility_feasibility_raw": self.safe_float(result.get("scenario_feasibility_score")),
                        "realism_feasibility_expl": self.safe_str(result.get("reasoning", ""))
                    })
                elif judge_name == "clarity_specificity":
                    output.update({
                        "clarity_specificity_clarity_raw": self.safe_float(result.get("clarity_score")),
                        "clarity_specificity_specificity_raw": self.safe_float(result.get("specificity_score")),
                        "clarity_specificity_expl": self.safe_str(result.get("reasoning", ""))
                    })
                elif judge_name == "follow_up_type":
                    output.update({
                        "follow_up_type_match_raw": self.safe_float(result.get("type_match_score")),
                        "follow_up_type_detected": self.safe_str(result.get("correct_follow_up_type", "")),
                        "follow_up_type_expl": self.safe_str(result.get("evaluation_reason", ""))
                    })
                elif judge_name == "safety_bias":
                    output.update({
                        "safety_bias_safety_raw": self.safe_float(result.get("safety_score")),
                        "safety_bias_bias_raw": self.safe_float(result.get("fairness_score")),
                        "safety_bias_expl": self.safe_str(result.get("explanation", ""))
                    })
                elif judge_name == "difficulty":
                    output.update({
                        "difficulty_raw": self.safe_float(result.get("difficulty_score")),
                        "difficulty_expl": self.safe_str(result.get("evaluation_reason", ""))
                    })
                elif judge_name == "intent_completeness":
                    output.update({
                        "intent_completeness_raw": self.safe_float(result.get("intent_score")),
                        "intent_completeness_expl": self.safe_str(result.get("reasoning", ""))
                    })
                else:
                    output.update({
                        f"{judge_name}_raw": self.safe_float(result.get("score")),
                        f"{judge_name}_expl": self.safe_str(result.get("reasoning", result.get("explanation", "")))
                    })

                return output

        except Exception as e:
            self.stats['failed_judgments'] += 1
            print(f" {judge_name} evaluation failed: {str(e)}")
            return {
                f"{judge_name}_raw": 0.0,
                f"{judge_name}_expl": f"Evaluation error: {str(e)}"
            }

    def evaluate_single_query(self, query_data: Dict[str, Any]) -> Dict[str, Any]:
        """Evaluate one query through all judges"""
        results = dict(query_data)  # Keep original data

        for judge_name in self.judge_names:
            judge_result = self.evaluate_single_judge(judge_name, query_data)
            results.update(judge_result)

            # Small delay between judges
            time.sleep(0.5)

        return results

    def evaluate_limited_queries(self, all_queries: List[Dict[str, Any]]) -> List[Dict[str, Any]]:
        """Evaluate only limited number of queries"""
        queries_to_process = all_queries[:QUERY_LIMIT]
        self.stats['total_queries'] = len(queries_to_process)

        print(f" PROCESSING {len(queries_to_process)} QUERIES (LIMITED)")
        print(f" Expected API calls: {len(queries_to_process)} queries × {len(self.judge_names)} judges = {len(queries_to_process) * len(self.judge_names)} calls")

        all_results = []

        for i, query_data in enumerate(tqdm(queries_to_process, desc="Evaluating queries")):
            query_num = i + 1
            print(f"\n Query {query_num}/{len(queries_to_process)}: {query_data.get('follow_up_query', '')[:80]}...")

            try:
                result = self.evaluate_single_query(query_data)
                all_results.append(result)

                # Save progress every 5 queries
                if query_num % 5 == 0:
                    self.save_progress(all_results, query_num)

                success_rate = (self.stats['successful_judgments'] / max(1, self.stats['successful_judgments'] + self.stats['failed_judgments'])) * 100
                print(f" Progress: {query_num}/{len(queries_to_process)} | Success: {success_rate:.1f}%")

            except Exception as e:
                print(f" Query {query_num} failed: {e}")
                all_results.append({**query_data, "error": f"Query processing failed: {e}"})

        return all_results

    def save_progress(self, results: List[Dict[str, Any]], query_num: int):
        """Save progress"""
        filename = f"progress_{query_num}_queries.json"

        output = {
            "metadata": {
                "queries_processed": query_num,
                "successful_judgments": self.stats['successful_judgments'],
                "failed_judgments": self.stats['failed_judgments'],
                "total_api_calls": self.stats['total_api_calls'],
                "timestamp": time.time(),
                "model": MODEL_NAME
            },
            "results": results
        }

        with open(filename, 'w') as f:
            json.dump(output, f, indent=2)

        print(f"💾 Progress saved: {filename}")

    def print_final_stats(self):
        """Print statistics"""
        total_time = time.time() - self.stats['start_time']

        print("\n" + "="*50)
        print(" FINAL STATISTICS")
        print("="*50)
        print(f"   Total Queries: {self.stats['total_queries']}")
        print(f"   Successful Judgments: {self.stats['successful_judgments']}")
        print(f"   Failed Judgments: {self.stats['failed_judgments']}")
        print(f"   Total API Calls: {self.stats['total_api_calls']}")
        print(f"   Total Time: {total_time:.1f}s ({total_time/60:.1f}m)")
        print(f"   Model: {MODEL_NAME}")

        if total_time > 0:
            qpm = (self.stats['total_queries'] / total_time) * 60
            print(f"   Speed: {qpm:.1f} queries/minute")

        success_rate = (self.stats['successful_judgments'] / max(1, self.stats['successful_judgments'] + self.stats['failed_judgments'])) * 100
        print(f"   Success Rate: {success_rate:.1f}%")

# ========= CALIBRATED EVALUATION SYSTEM =========
class CalibratedEvaluationSystem(OptimizedEvaluationSystem):
    def __init__(self, api_keys: List[str], judges_config: Dict[str, str]):
        super().__init__(api_keys, judges_config)
        self.calibration_system = CalibrationSystem()
        print(" CALIBRATED EVALUATION SYSTEM READY")

    def evaluate_with_calibration(self, all_queries: List[Dict], gold_queries: List[Dict] = None) -> List[Dict]:
        """Evaluate queries with optional calibration"""

        # If calibration enabled and gold queries provided, do calibration first
        if ENABLE_CALIBRATION and gold_queries and not self.calibration_system.is_calibrated:
            print(" STARTING CALIBRATION WITH GOLD DATASET...")
            calibration_metrics = self.calibration_system.calibrate_with_gold_data(gold_queries, self)

            # Save calibration verification
            self.calibration_system.save_calibration_verification()

            print("\n CALIBRATION RESULTS:")
            for metric, metrics in calibration_metrics.items():
                if metrics:  # Check if metrics dict is not empty
                    print(f"   {metric}: Corr={metrics.get('correlation', 0):.3f}, Bias={metrics.get('bias', 0):.3f}, MAE={metrics.get('mae', 0):.3f}")

        # Evaluate all queries
        results = []
        for query in tqdm(all_queries, desc="Evaluating queries"):
            raw_score = self.evaluate_single_query(query)

            # Apply calibration if available and enabled
            if ENABLE_CALIBRATION and self.calibration_system.is_calibrated:
                calibrated_score = self.calibration_system.apply_calibration(raw_score)
                results.append(calibrated_score)
            else:
                results.append(raw_score)

        return results

# ========= MAIN WORKFLOW =========
def run_calibrated_workflow(test_file: str = "test.json", gold_file: str = "gold.json"):
    """Run workflow with optional calibration"""

    print(" CALIBRATED EVALUATION WORKFLOW")
    print("="*60)
    print(f" CALIBRATION: {'ENABLED' if ENABLE_CALIBRATION else 'DISABLED'}")
    print(f" PROCESSING ONLY {QUERY_LIMIT} QUERIES")
    print(f" USING MODEL: {MODEL_NAME}")

    # Test API keys first
    working_keys = test_api_keys()

    if not working_keys:
        print(" NO WORKING API KEYS FOUND!")
        return

    print(f" Using {len(working_keys)} working API keys")

    # Load evaluation queries
    print("\n Loading evaluation queries...")
    try:
        with open(test_file, 'r') as f:
            test_data = json.load(f)

        # Updated for Task 2 schema
        evaluation_queries = test_data.get("task2_queries", [])[:QUERY_LIMIT]
        print(f" Found {len(evaluation_queries)} evaluation queries")

    except Exception as e:
        print(f" Failed to load {test_file}: {e}")
        return

    # Load gold queries for calibration if enabled
    gold_queries = []
    if ENABLE_CALIBRATION and gold_file:
        try:
            with open(gold_file, 'r') as f:
                gold_data = json.load(f)
            gold_queries = gold_data.get("results", [])
            print(f" Gold queries for calibration: {len(gold_queries)}")
        except Exception as e:
            print(f" Failed to load gold file: {e}")
            print(" Continuing without calibration...")

    # Initialize system
    print(" Initializing evaluation system...")
    if ENABLE_CALIBRATION:
        evaluator = CalibratedEvaluationSystem(
            api_keys=working_keys,
            judges_config=JUDGES_CONFIG
        )
    else:
        evaluator = OptimizedEvaluationSystem(
            api_keys=working_keys,
            judges_config=JUDGES_CONFIG
        )

    # Run evaluation
    print("\n STARTING EVALUATION")
    print("-" * 40)

    if ENABLE_CALIBRATION:
        results = evaluator.evaluate_with_calibration(evaluation_queries, gold_queries)
    else:
        results = evaluator.evaluate_limited_queries(evaluation_queries)

    # Save final results
    timestamp = int(time.time())
    calibration_status = "calibrated" if ENABLE_CALIBRATION and evaluator.calibration_system.is_calibrated else "uncalibrated"
    output_file = f"task2_{calibration_status}_results_{QUERY_LIMIT}_{timestamp}.json"

    with open(output_file, 'w') as f:
        json.dump({
            "metadata": {
                "total_queries": len(results),
                "timestamp": timestamp,
                "model": MODEL_NAME,
                "calibration_applied": ENABLE_CALIBRATION and evaluator.calibration_system.is_calibrated if ENABLE_CALIBRATION else False,
                "calibration_params": evaluator.calibration_system.calibration_params if ENABLE_CALIBRATION and evaluator.calibration_system.is_calibrated else {},
                "success_rate": f"{(evaluator.stats['successful_judgments'] / max(1, evaluator.stats['successful_judgments'] + evaluator.stats['failed_judgments'])) * 100:.1f}%"
            },
            "results": results
        }, f, indent=2)

    evaluator.print_final_stats()
    print(f"\n EVALUATION COMPLETED!")
    print(f" Results saved to: {output_file}")

# ========= RUN THE WORKFLOW =========
if __name__ == "__main__":
    run_calibrated_workflow(
        test_file="merged_task2_queries.json",  # Your Task 2 queries file
        gold_file="gold.json"  #gold data available
    )